# DocuSense - Notebook 1: build the datasets
**Accelerator: None (CPU)** - saves GPU quota. **Internet: ON.**
**Add Data**: your `docusense-code` dataset + the Kaggle dataset **SROIE datasetv2** (search "SROIE datasetv2").
Runtime ~30-60 min. When done: **Save Version -> Save & Run All (Commit)**, then Output tab ->
**New Dataset** -> name it `docusense-data`. Notebooks 2-4 attach that dataset.

In [ ]:
import glob, os, sys, random, json
import numpy as np

hits = sorted(glob.glob("/kaggle/input/**/models/tagger/labels.py", recursive=True), key=len)   # any nesting depth
if not hits:   # show what IS attached, so the problem is visible
    for dp, ds, fs in os.walk("/kaggle/input"):
        if dp.count("/") <= 5: print(dp, ds[:6], fs[:4])

assert hits, "Code dataset not found: add your `docusense-code` dataset via Add Data (see KAGGLE_TRAINING.md)"
CODE = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
sys.path.insert(0, CODE)
print("code from:", CODE)

os.system("pip install -q datasets")            # Hugging Face `datasets` (data only - no models)
random.seed(42); np.random.seed(42)
WORK = "/kaggle/working/data"
os.makedirs(WORK, exist_ok=True)

In [ ]:
from data.build_cord import build_cord
cord_stats = build_cord(WORK, hf_id="naver-clova-ix/cord-v2", audit=20)

In [ ]:
from data.build_sroie import build_sroie
# the DATASET "SROIE datasetv2" (by urbikn) - not a notebook. Its folder holds train/ and test/ with images + box/entities txt files.
cands = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/*.jpg", recursive=True)
                if "sroie" in p.lower() and "layoutlm" not in p.lower()}, key=len)
assert cands, "SROIE dataset not attached: Add Data -> Datasets -> search 'SROIE datasetv2' (the DATASET, not the LayoutLM notebook)"
SROIE_ROOT = os.path.dirname(cands[0]) if os.path.basename(cands[0]).lower() in ("train", "test") else cands[0]
print("SROIE root:", SROIE_ROOT)
sroie_res = build_sroie(WORK, SROIE_ROOT, audit=20)
# GATE: coverage (docs with date + total found) must be >= 95%. Vendor may be missing on some docs - those are kept with
# VENDOR marked 'not annotated'. Check the audit images below.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
imgs = sorted(glob.glob(f"{WORK}/audit/*.jpg"))
pick = imgs[:4] + imgs[len(imgs)//2: len(imgs)//2 + 4]
fig, axes = plt.subplots(2, 4, figsize=(20, 12))
for ax, p in zip(axes.ravel(), pick):
    ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()
# Check: vendor (red), date (blue), items (green/orange/purple), subtotal/tax/total sit on the right words.

In [ ]:
from data.synth_invoices import build_synth
build_synth(WORK, n_train=2000, n_val=200, audit=10)

In [ ]:
from data.build_rvl_cache import build_cls_cache
# Tries these Hugging Face copies in order (classes are mapped by NAME, so different id orders don't matter).
# If they all fail (id/format changed), attach a local RVL-CDIP copy (one sub-folder per class, e.g. invoice/, form/, letter/,
# email/, memo/, resume/, news_article/) and pass rvl_dir="/kaggle/input/<that-dataset>" instead.
stats = build_cls_cache(WORK, quotas={"train": 3000, "val": 500, "test": 500}, hf_ids=("chainyo/rvl-cdip",))

In [ ]:
import subprocess
print(subprocess.run(["bash", "-c", f"ls -lh {WORK} | head -60; du -sh {WORK}"], capture_output=True, text=True).stdout)
print(json.load(open(f"{WORK}/cls_stats.json")))
# Now: Save Version -> Save & Run All (Commit).  Then Output tab -> New Dataset -> `docusense-data`.